# model_artifacts_2 — unit test

Tests the active model pointer and date tracking without needing a trained model.

In [1]:
import sys
sys.path.insert(0, '..')

import src.models.model_artifacts_2 as ma2
from src.config.settings import get_settings

settings = get_settings()
print('local_artifacts_path:', settings.local_artifacts_path)
print('model dir           :', ma2._model_dir(settings))

local_artifacts_path: artifacts
model dir           : artifacts/model


In [2]:
# Test 1: init_model_dirs creates expected structure
ma2.init_model_dirs(settings)

from pathlib import Path
base = ma2._model_dir(settings)

assert (base / 'model_a').exists(), 'model_a dir missing'
assert (base / 'model_b').exists(), 'model_b dir missing'
assert (base / 'active_model').exists(), 'active_model file missing'

print('active_model file contains:', (base / 'active_model').read_text())
print('PASS: directory structure correct')

active_model file contains: model_a
PASS: directory structure correct


In [3]:
# Test 2: get_active_model reads from file correctly
# Force reset the in-memory variable to simulate a fresh process
ma2.ACTIVE_MODEL = None

active = ma2.get_active_model(settings)
print('get_active_model() ->', active)
assert active in ma2.SLOTS, f'unexpected slot: {active}'
assert ma2.ACTIVE_MODEL == active, 'in-memory variable not set after first call'
print('PASS: reads from disk on first call, caches in memory')

get_active_model() -> model_a
PASS: reads from disk on first call, caches in memory


In [4]:
# Test 3: inactive_model returns the other slot
active = ma2.get_active_model(settings)
inactive = ma2.inactive_model(settings)

print(f'active={active}  inactive={inactive}')
assert active != inactive, 'active and inactive must differ'
assert inactive in ma2.SLOTS
print('PASS: inactive_model returns the other slot')

active=model_a  inactive=model_b
PASS: inactive_model returns the other slot


In [5]:
# Test 4: update_active_model switches slot + writes to disk + updates in-memory
original = ma2.get_active_model(settings)
target   = ma2.inactive_model(settings)

ma2.update_active_model(target, settings)

assert ma2.ACTIVE_MODEL == target,                           'in-memory not updated'
assert (base / 'active_model').read_text().strip() == target, 'file not updated'
assert ma2.get_active_model(settings) == target,             'get_active_model disagrees'

print(f'Switched: {original} -> {target}')
print('active_model file now contains:', (base / 'active_model').read_text())
print('PASS: update writes to memory and disk')

Switched: model_a -> model_b
active_model file now contains: model_b
PASS: update writes to memory and disk


In [6]:
# Test 5: simulate process restart — reset in-memory, read back from disk
ma2.ACTIVE_MODEL = None
recovered = ma2.get_active_model(settings)

assert recovered == target, f'expected {target}, got {recovered}'
print(f'After simulated restart, get_active_model() -> {recovered}')
print('PASS: survives process restart via disk read')

After simulated restart, get_active_model() -> model_b
PASS: survives process restart via disk read


In [7]:
# Test 6: write_dates / read_dates round-trip
ma2.write_dates('istdaten_dates', start='2025-01-01', end='2026-01-31', settings=settings)
ma2.write_dates('model_trained_dates', start='2025-01-01', end='2026-01-24', settings=settings)

id_dates = ma2.read_dates('istdaten_dates', settings)
mt_dates = ma2.read_dates('model_trained_dates', settings)

print('istdaten_dates     :', id_dates)
print('model_trained_dates:', mt_dates)
print()
print((base / 'istdaten_dates').read_text())
print((base / 'model_trained_dates').read_text())

assert id_dates == {'START': '2025-01-01', 'END': '2026-01-31'}
assert mt_dates == {'START': '2025-01-01', 'END': '2026-01-24'}
print('PASS: write_dates / read_dates round-trip correct')

istdaten_dates     : {'START': '2025-01-01', 'END': '2026-01-31'}
model_trained_dates: {'START': '2025-01-01', 'END': '2026-01-24'}

START=2025-01-01
END=2026-01-31

START=2025-01-01
END=2026-01-24

PASS: write_dates / read_dates round-trip correct


In [8]:
# Test 7: read_dates on missing file returns {}
result = ma2.read_dates('nonexistent_file', settings)
assert result == {}, f'expected empty dict, got {result}'
print('PASS: read_dates returns {} for missing file')

PASS: read_dates returns {} for missing file


In [9]:
# Test 8: update_active_model rejects invalid slot names
try:
    ma2.update_active_model('model_c', settings)
    assert False, 'should have raised ValueError'
except ValueError as e:
    print(f'Correctly rejected invalid slot: {e}')
print('PASS: invalid slot raises ValueError')

Correctly rejected invalid slot: slot must be one of ('model_a', 'model_b'), got 'model_c'
PASS: invalid slot raises ValueError


In [10]:
# Restore active model to model_a so directory is in a clean state
ma2.update_active_model('model_a', settings)
print('Restored active_model to model_a')
print()
print('=== All tests passed ===')

Restored active_model to model_a

=== All tests passed ===
